# Transcription & Phonemic Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensein/senselab/blob/main/tutorials/audio/transcription_and_phonemic_analysis.ipynb)

In this tutorial you will learn to:
- Transcribe speech using automatic speech recognition (Whisper)
- Align transcriptions to audio at the word and phoneme level
- Extract phonetic posteriorgrams (PPGs) and analyze phoneme durations
- Visualize phoneme boundaries aligned with waveforms and spectrograms

In [ ]:
# Install senselab and dependencies
!pip install -q uv
!uv pip install --pre --system "senselab[nlp]"

> **\u26a0\ufe0f Restart runtime after install**
>
> The install may upgrade packages (e.g., numpy) that are already loaded.
> Go to **Runtime \u2192 Restart session**, then **run all cells below** (skip this install cell).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from senselab.audio.data_structures import Audio
from senselab.audio.tasks.features_extraction.ppg import (
    extract_mean_phoneme_durations,
    extract_ppgs_from_audios,
    plot_ppg_phoneme_timeline,
)
from senselab.audio.tasks.forced_alignment.forced_alignment import align_transcriptions
from senselab.audio.tasks.plotting.plotting import play_audio, plot_waveform_and_specgram
from senselab.audio.tasks.preprocessing import downmix_audios_to_mono, resample_audios
from senselab.audio.tasks.speech_to_text.api import transcribe_audios
from senselab.utils.data_structures import DeviceType, HFModel, Language, ScriptLine

device = DeviceType.CUDA if torch.cuda.is_available() else DeviceType.CPU
print(f"Using device: {device.value}")

## 1. Load Audio

We need a clear English speech sample. We will download a test file from the senselab repository,
resample it to 16 kHz (required by both the ASR and alignment models), and ensure it is mono.

In [ ]:
import urllib.request

os.makedirs("tutorial_audio_files", exist_ok=True)
urllib.request.urlretrieve(
    "https://github.com/sensein/senselab/raw/main/src/tests/data_for_testing/audio_48khz_mono_16bits.wav",
    "tutorial_audio_files/sample_speech.wav",
)

audio = Audio(filepath=os.path.abspath("tutorial_audio_files/sample_speech.wav"))
audio_16k = resample_audios([audio], resample_rate=16000)[0]
print(f"Duration: {audio_16k.waveform.shape[1] / audio_16k.sampling_rate:.2f}s at {audio_16k.sampling_rate} Hz")
play_audio(audio_16k)

## 2. Automatic Speech Recognition

Automatic speech recognition (ASR) converts speech to text. We use OpenAI's Whisper,
a transformer model trained on 680,000 hours of multilingual audio. It takes audio as
input and produces a text transcription.

Senselab wraps the HuggingFace Transformers pipeline, so all you need to do is specify
the model and the audio.

In [ ]:
model = HFModel(path_or_uri="openai/whisper-small", revision="main")
transcripts = transcribe_audios(audios=[audio_16k], model=model, device=device)
transcript_text = transcripts[0].text
print(f"Transcription: {transcript_text}")

## 3. Forced Alignment

Given audio and its transcription, **forced alignment** finds the precise time boundaries
for each word and each phoneme (character). This uses a CTC-based wav2vec2 model that
learns the correspondence between audio frames and text characters.

The alignment result is a nested `ScriptLine` hierarchy:
- **Utterance** level: the entire transcription with start/end times
- **Sentence** level: sentences within the utterance
- **Word** level: individual words with timestamps
- **Character** level: individual characters (phonemes) with timestamps

In [ ]:
script = ScriptLine(text=transcript_text)
alignment_results = align_transcriptions(
    [(audio_16k, script, Language(language_code="en"))],
    levels_to_keep={"utterance": True, "word": True, "char": True},
)

# The result is a list (per audio) of lists (per segment) of ScriptLine objects.
# With utterance=True, we get the full utterance with nested sentences/words/chars.
utterance = alignment_results[0][0]

# Collect word-level items by walking the hierarchy:
# utterance -> sentences -> words -> chars
words = []
if utterance and utterance.chunks:
    for sentence in utterance.chunks:
        if sentence.chunks:
            for word in sentence.chunks:
                words.append(word)

print("=== Word-level Alignment ===")
for word in words:
    print(f"  [{word.start:.3f}s - {word.end:.3f}s] {word.text}")

## 4. Aligned Visualization

Visualizing word boundaries alongside the waveform and spectrogram gives a complete picture
of speech. You can see which words correspond to which acoustic patterns.

Below we create a three-panel aligned plot:
- **Top**: Waveform (amplitude over time)
- **Middle**: Mel spectrogram (frequency content over time)
- **Bottom**: Word boundaries as colored spans with character-level tick marks

In [ ]:
from senselab.audio.tasks.features_extraction.torchaudio import extract_mel_spectrogram_from_audios

waveform = audio_16k.waveform.squeeze().numpy()
sr = audio_16k.sampling_rate
duration = len(waveform) / sr
time_axis = np.linspace(0, duration, len(waveform))

# Extract mel spectrogram using senselab
mel_result = extract_mel_spectrogram_from_audios([audio_16k], n_fft=1024, hop_length=256, n_mels=80)
spec = mel_result[0]["mel_spectrogram"]

# Convert to dB scale
spec_db = 10 * torch.log10(spec.clamp(min=1e-10))
spec_db = spec_db - spec_db.max()  # normalize

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [1, 2, 1]})

# Panel 1: Waveform
axes[0].plot(time_axis, waveform, linewidth=0.3, color="steelblue")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Aligned Speech Analysis: Waveform + Spectrogram + Word Boundaries")
axes[0].grid(True, alpha=0.2)

# Panel 2: Mel Spectrogram
axes[1].imshow(
    spec_db.numpy(),
    aspect="auto",
    origin="lower",
    extent=[0, duration, 0, 8],
    cmap="magma",
)
axes[1].set_ylabel("Frequency (kHz)")

# Panel 3: Word and character boundaries
if words:
    colors = plt.get_cmap("Set3", max(len(words), 3))
    for i, word in enumerate(words):
        if word.start is not None and word.end is not None:
            axes[2].axvspan(word.start, word.end, alpha=0.3, color=colors(i % colors.N))
            axes[2].text(
                (word.start + word.end) / 2,
                0.5,
                word.text,
                ha="center",
                va="center",
                fontsize=7,
                fontweight="bold",
            )
            # Add character-level boundaries if available
            if word.chunks:
                for char in word.chunks:
                    if char.start is not None:
                        axes[2].axvline(char.start, color="gray", linewidth=0.5, alpha=0.5)

axes[2].set_ylabel("Words")
axes[2].set_xlabel("Time (seconds)")
axes[2].set_xlim(0, duration)
axes[2].set_yticks([])

plt.tight_layout()
plt.show()

## 5. Phonetic Posteriorgrams (PPGs)

While forced alignment gives binary phoneme boundaries, **Phonetic Posteriorgrams (PPGs)**
provide a richer view: the probability of **each** phoneme at every time frame.

This captures the continuous, overlapping nature of speech sounds (coarticulation). PPGs
are extracted by a neural network that maps audio frames to phoneme probability distributions.

Note: PPG extraction uses an isolated subprocess virtual environment (because the `ppgs` library
has specific dependency requirements). The first run may take a few minutes to set up.

In [ ]:
ppgs = extract_ppgs_from_audios([audio_16k], device=device)
ppg = ppgs[0]
print(f"PPG shape: {ppg.shape}")
print("This gives us the probability of each phoneme at every time frame.")

## 6. Phoneme Duration Analysis

From the PPG, we can determine which phoneme is most likely at each frame (argmax),
group contiguous frames into segments, and calculate how long each phoneme lasts.

This reveals speaking rhythm, articulation speed, and phonemic patterns. The analysis
aggregates across all segments of the same phoneme to compute mean and total durations.

In [ ]:
durations = extract_mean_phoneme_durations(audio_16k, ppg)
print(f"Analysis: {durations['frame_count']} frames over {durations['analysis_duration_seconds']:.2f}s")
print("\n=== Top 10 Phonemes by Total Duration ===")
for p in sorted(durations["phoneme_durations"], key=lambda x: x["total_duration_seconds"], reverse=True)[:10]:
    bar = "\u2588" * int(p["total_duration_seconds"] * 50)
    print(f"  {p['phoneme']:>10s}: {p['mean_duration_seconds'] * 1000:6.1f}ms avg ({p['count']} segments) {bar}")

In [ ]:
plot_ppg_phoneme_timeline(audio_16k, ppg, title="Phoneme Timeline", show=True);

## Summary

In this tutorial we covered the full pipeline from raw audio to detailed phonemic analysis:

1. **ASR with Whisper**: Transcribed speech to text using a pre-trained transformer model.
2. **Forced alignment**: Mapped words and characters to precise time boundaries using wav2vec2.
3. **Multi-panel visualization**: Combined waveform, spectrogram, and word boundaries on a shared time axis.
4. **PPG extraction**: Obtained continuous phoneme probability distributions across time.
5. **Phoneme duration analysis**: Computed per-phoneme statistics revealing articulation patterns.

For more tutorials, see the [senselab tutorials directory](https://github.com/sensein/senselab/tree/main/tutorials).